# Cart-pole direct transcription in PyDrake, with SNOPT and Meshcat rollout

This notebook mirrors the MATLAB/SNOPT direct-transcription formulation, but keeps the model inside Drake:

\[
    w = [h, X(:), U(:)], \qquad x_{k+1} - \phi_h(x_k,u_k)=0.
\]

The important choices here are:

- `h` is a true decision variable with bounds.
- Dynamics constraints use Drake's `Simulator.AdvanceTo(...)`, not forward Euler.
- The simulator integrator is configured through PyDrake's `ResetIntegratorFromFlags(...)` path.
- SNOPT is called explicitly instead of relying on Drake's automatic solver selection.
- After solving, the optimized zero-order-hold input is rolled out again in Meshcat.

The 0.02--0.05 final error you saw before was mostly because the terminal box was intentionally loose and because open-loop rollout with a discretized input can drift from the NLP knot states. This version tightens the terminal box and uses multiple RK2 substeps per transcription interval.


In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from pydrake.all import (
    AddMultibodyPlantSceneGraph,
    AutoDiffXd,
    ChooseBestSolver,
    DiagramBuilder,
    InitializeParams,
    LogVectorOutput,
    MathematicalProgram,
    Parser,
    PiecewisePolynomial,
    ResetIntegratorFromFlags,
    Simulator,
    SnoptSolver,
    StartMeshcat,
    TrajectorySource,
)
from pydrake.visualization import AddDefaultVisualization

from underactuated import ConfigureParser


## Build the optimization plant from Drake's URDF

This follows the same construction style as the cart-pole balancing notebook: `DiagramBuilder`, `AddMultibodyPlantSceneGraph`, `Parser`, `ConfigureParser`, and `package://underactuated/models/undamped_cartpole.urdf`.


In [ ]:
def make_cartpole_diagram():
    builder = DiagramBuilder()
    cartpole, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.0)
    parser = Parser(cartpole)
    ConfigureParser(parser)
    parser.AddModelsFromUrl("package://underactuated/models/undamped_cartpole.urdf")
    cartpole.Finalize()
    return builder, cartpole, scene_graph

# Plant used inside the direct-transcription dynamics constraint.
builder, cartpole, scene_graph = make_cartpole_diagram()
diagram = builder.Build()

# State convention used by the underactuated cart-pole notebooks:
# x = [cart position, pole angle, cart velocity, pole angular velocity].
x_initial = np.array([0.0, 0.0, 0.0, 0.0])
x_target = np.array([0.0, np.pi, 0.0, 0.0])

# Quick sanity check at the upright equilibrium.
context = diagram.CreateDefaultContext()
plant_context = cartpole.GetMyMutableContextFromRoot(context)
plant_context.get_mutable_continuous_state_vector().SetFromVector(x_target)
cartpole.get_actuation_input_port().FixValue(plant_context, [0.0])


## Drake-integrated one-step flow map

The dynamics defect is

\[
    g_k = x_{k+1} - \phi_h(x_k,u_k),
\]

where \(\phi_h\) is obtained by simulating the Drake cart-pole model over one interval while holding \(u_k\) constant.

I use `runge_kutta2` because PyDrake documents it as a fixed-step integrator. To reduce the transcription/rollout mismatch, each NLP interval is integrated with several RK2 substeps by setting

\[
    \Delta t_\text{internal} = h / n_\text{substeps}.
\]

This is still deterministic for the callback, but much more accurate than one RK2 step or Euler.


In [ ]:
# Number of fixed RK2 substeps used inside each direct-transcription interval.
# Increase to 10 if the rollout error is still visibly too large.
integration_substeps = 6
integrator_scheme = "runge_kutta2"


def _is_autodiff_vector(z):
    return len(z) > 0 and isinstance(z[0], AutoDiffXd)


def _value_vector(z):
    if _is_autodiff_vector(z):
        return np.array([zi.value() for zi in z], dtype=float)
    return np.asarray(z, dtype=float)


def _derivative_matrix(z):
    return np.vstack([zi.derivatives() for zi in z])


def _as_autodiff_vector(y_value, local_jacobian, z_ad):
    # z_ad has derivatives dz/dw, where w is the full MathematicalProgram
    # decision vector. Chain rule gives dy/dw = dy/dz * dz/dw.
    dz_dw = _derivative_matrix(z_ad)
    dy_dw = local_jacobian @ dz_dw
    return np.array(
        [AutoDiffXd(y_value[i], dy_dw[i, :]) for i in range(len(y_value))],
        dtype=object,
    )


def integrate_cartpole_one_step_value(xk, uk, hk, substeps=integration_substeps):
    """Return phi_h(xk, uk) using PyDrake Simulator.AdvanceTo.

    The input is held constant over [0, h]. The simulator uses Drake's
    fixed-step runge_kutta2 integrator with max_step_size = h / substeps.
    """
    hk = float(hk)
    uk = np.asarray(uk, dtype=float).reshape(1)
    xk = np.asarray(xk, dtype=float).reshape(4)

    local_context = diagram.CreateDefaultContext()
    local_plant_context = cartpole.GetMyMutableContextFromRoot(local_context)

    local_context.SetTime(0.0)
    local_plant_context.get_mutable_continuous_state_vector().SetFromVector(xk)
    cartpole.get_actuation_input_port().FixValue(local_plant_context, uk)

    simulator = Simulator(diagram, local_context)
    ResetIntegratorFromFlags(simulator, integrator_scheme, hk / substeps)

    init = InitializeParams()
    init.suppress_initialization_events = True
    simulator.Initialize(init)
    simulator.AdvanceTo(hk, interruptible=False)

    final_context = simulator.get_context()
    final_plant_context = cartpole.GetMyContextFromRoot(final_context)
    return final_plant_context.get_continuous_state_vector().CopyToVector()


def _finite_difference_jacobian(fun, z, eps=1e-7):
    z = np.asarray(z, dtype=float)
    y0 = np.asarray(fun(z), dtype=float)
    J = np.zeros((len(y0), len(z)))
    for j in range(len(z)):
        step = eps * (1.0 + abs(z[j]))
        zp = z.copy()
        zm = z.copy()
        zp[j] += step
        zm[j] -= step
        J[:, j] = (np.asarray(fun(zp)) - np.asarray(fun(zm))) / (2.0 * step)
    return J


def integrate_cartpole_one_step(xk, uk, hk):
    return integrate_cartpole_one_step_value(xk, uk, hk)


print("upright one-step test:", integrate_cartpole_one_step(x_target, [0.0], 0.05))


## Direct transcription problem

This uses `MathematicalProgram` directly, like the double-integrator notebook, but with the cart-pole flow map coming from Drake.

The terminal box is now tight. If SNOPT has trouble from a cold start, first use `terminal_tol = [5e-3, 5e-3, 1e-2, 1e-2]`, solve once, then warm-start the tighter version.


In [ ]:
# More knots reduces open-loop rollout mismatch. 81 is a good first serious run.
# Increase to 121 or 161 after the notebook is working on your machine.
N = 81
nX = 4
nU = 1

# h is a true NLP decision variable.
h_min = 0.02
h_max = 0.06
h_guess = 0.04

x_min = np.array([-10.0, -10.0, -10.0, -10.0])
x_max = np.array([ 10.0,  10.0,  10.0,  10.0])
u_min = np.array([-100.0])
u_max = np.array([ 100.0])

# Tight terminal box. This directly removes the previous 0.02--0.05 tolerance-level error.
terminal_tol = np.array([1e-3, 1e-3, 5e-3, 5e-3])
xf_min = x_target - terminal_tol
xf_max = x_target + terminal_tol

Q = np.diag([0.01, 0.01, 0.01, 0.01])
Qf = np.diag([10.0, 200.0, 10.0, 10.0])
R = np.diag([0.01])

# Small final-time pressure. Set this to 0 for the pure MATLAB-style quadratic cost.
time_weight = 0.1


In [ ]:
def quadratic_form(z, Q):
    return sum(Q[i, j] * z[i] * z[j] for i in range(Q.shape[0]) for j in range(Q.shape[1]))


def _defect_value(z):
    xk = z[0:4]
    uk = z[4:5]
    xkp1 = z[5:9]
    hk = z[9]
    return xkp1 - integrate_cartpole_one_step_value(xk, uk, hk)


def dynamics_defect(z):
    """Return x_{k+1} - phi_h(x_k, u_k).

    MathematicalProgram may evaluate this callback with AutoDiffXd. Since the
    flow map is a simulator-based numerical black box, we supply local finite-
    difference derivatives and return AutoDiffXd outputs by the chain rule.
    """
    if _is_autodiff_vector(z):
        z_value = _value_vector(z)
        y_value = _defect_value(z_value)
        J = _finite_difference_jacobian(_defect_value, z_value)
        return _as_autodiff_vector(y_value, J, z)
    return _defect_value(z)


In [ ]:
prog = MathematicalProgram()

# Decision variables: X is 4 x N, U is 1 x (N-1), h is scalar.
x = prog.NewContinuousVariables(nX, N, "x")
u = prog.NewContinuousVariables(nU, N - 1, "u")
h = prog.NewContinuousVariables(1, "h")[0]

# Initial and terminal state constraints.
prog.AddBoundingBoxConstraint(x_initial, x_initial, x[:, 0])
prog.AddBoundingBoxConstraint(xf_min, xf_max, x[:, -1])

# Bounds on h, states, and inputs.
prog.AddBoundingBoxConstraint(h_min, h_max, np.array([h]))
for k in range(N):
    prog.AddBoundingBoxConstraint(x_min, x_max, x[:, k])
for k in range(N - 1):
    prog.AddBoundingBoxConstraint(u_min, u_max, u[:, k])

# Dynamics constraints.
for k in range(N - 1):
    vars_k = np.concatenate((x[:, k], u[:, k], x[:, k + 1], np.array([h])))
    prog.AddConstraint(dynamics_defect, np.zeros(nX), np.zeros(nX), vars_k)

# Running cost: h * sum_k [(x_k - xd)' Q (x_k - xd) + u_k' R u_k].
for k in range(N - 1):
    e = x[:, k] - x_target
    prog.AddCost(h * (quadratic_form(e, Q) + quadratic_form(u[:, k], R)))

# Terminal cost.
eN = x[:, -1] - x_target
prog.AddCost(quadratic_form(eN, Qf))

# Optional final-time pressure.
prog.AddCost(time_weight * (N - 1) * h)

# SNOPT options. These mirror the spirit of the MATLAB SNOPT version.
prog.SetSolverOption(SnoptSolver.id(), "Major Feasibility Tolerance", 1e-8)
prog.SetSolverOption(SnoptSolver.id(), "Major Optimality Tolerance", 1e-8)
prog.SetSolverOption(SnoptSolver.id(), "Minor Feasibility Tolerance", 1e-8)
prog.SetSolverOption(SnoptSolver.id(), "Iterations Limit", 10000)
prog.SetSolverOption(SnoptSolver.id(), "Major Iterations Limit", 10000)

# Initial guess: linear state interpolation, zero controls, middle of h bounds.
for i in range(nX):
    prog.SetInitialGuess(x[i, :], np.linspace(x_initial[i], x_target[i], N))
prog.SetInitialGuess(u, np.zeros((nU, N - 1)))
prog.SetInitialGuess(h, h_guess)

print("Drake would automatically choose:", ChooseBestSolver(prog).name())


## Solve explicitly with SNOPT

`Solve(prog)` asks Drake to pick a solver. Here we call SNOPT directly so there is no ambiguity.


In [ ]:
snopt = SnoptSolver()
print("SNOPT available:", snopt.available())
print("SNOPT enabled:", snopt.enabled())

result = snopt.Solve(prog)

print("solver used:", result.get_solver_id().name())
print("success:", result.is_success())
print("solution result:", result.get_solution_result())
try:
    details = result.get_solver_details()
    print("SNOPT info:", details.info)
except Exception:
    pass

h_sol = result.GetSolution(h)
print("optimal h =", h_sol)
print("total time =", (N - 1) * h_sol)


In [ ]:
x_sol = result.GetSolution(x)
u_sol = result.GetSolution(u)
h_sol = result.GetSolution(h)

tx = np.arange(N) * h_sol
tu = np.arange(N - 1) * h_sol

print("transcription final state =", x_sol[:, -1])
print("terminal error =", x_sol[:, -1] - x_target)
print("terminal error inf-norm =", np.linalg.norm(x_sol[:, -1] - x_target, ord=np.inf))


In [ ]:
plt.figure()
plt.plot(x_sol[0, :], x_sol[1, :], "-o", markersize=2)
plt.plot(x_initial[0], x_initial[1], "go", label="start")
plt.plot(x_target[0], x_target[1], "ro", label="target")
plt.xlabel("cart position")
plt.ylabel("pole angle")
plt.title("Cart-pole swing-up trajectory")
plt.grid(True)
plt.legend()

plt.figure()
plt.plot(tx, x_sol[0, :])
plt.xlabel("time [s]")
plt.ylabel("cart position")
plt.grid(True)

plt.figure()
plt.plot(tx, x_sol[1, :])
plt.xlabel("time [s]")
plt.ylabel("pole angle")
plt.grid(True)

plt.figure()
plt.step(tu, u_sol[0, :], where="post")
plt.xlabel("time [s]")
plt.ylabel("cart force")
plt.grid(True)

plt.figure()
plt.plot(x_sol[1, :], x_sol[3, :])
plt.xlabel("pole angle")
plt.ylabel("angular velocity")
plt.title("Angle phase portrait")
plt.grid(True)
plt.show()


## Check every optimized defect

This confirms whether the NLP constraints themselves are tight. If these are small but the Meshcat rollout drifts, the issue is rollout/input discretization rather than constraint feasibility.


In [ ]:
def all_defects(x_traj, u_traj, h_value):
    defects = np.zeros((nX, N - 1))
    for k in range(N - 1):
        x_next = integrate_cartpole_one_step_value(x_traj[:, k], u_traj[:, k], h_value)
        defects[:, k] = x_traj[:, k + 1] - x_next
    return defects

D = all_defects(x_sol, u_sol, h_sol)
print("max abs defect =", np.max(np.abs(D)))
print("max defect 2-norm across intervals =", np.max(np.linalg.norm(D, axis=0)))


## Roll out the optimized open-loop input in Meshcat

The optimized control sequence is used as a zero-order-hold trajectory:

\[
    u(t) = u_k, \qquad t \in [k h, (k+1)h).
\]

This is the real sanity check: the NLP states are only knot variables, while the rollout is an independent continuous simulation driven by the optimized input.


In [ ]:
meshcat = StartMeshcat()
meshcat.Delete()

# Duplicate the last input sample so ZeroOrderHold has the same number of breaks and samples.
rollout_times = np.arange(N) * h_sol
u_samples = np.hstack([u_sol, u_sol[:, -1:]])
u_traj = PiecewisePolynomial.ZeroOrderHold(rollout_times, u_samples)

builder = DiagramBuilder()
cartpole_sim, scene_graph_sim = AddMultibodyPlantSceneGraph(builder, time_step=0.0)
parser = Parser(cartpole_sim)
ConfigureParser(parser)
parser.AddModelsFromUrl("package://underactuated/models/undamped_cartpole.urdf")
cartpole_sim.Finalize()

u_source = builder.AddSystem(TrajectorySource(u_traj))
builder.Connect(u_source.get_output_port(), cartpole_sim.get_actuation_input_port())

state_logger = LogVectorOutput(cartpole_sim.get_state_output_port(), builder)
AddDefaultVisualization(builder, meshcat)

sim_diagram = builder.Build()
sim_context = sim_diagram.CreateDefaultContext()
sim_plant_context = cartpole_sim.GetMyMutableContextFromRoot(sim_context)
sim_plant_context.get_mutable_continuous_state_vector().SetFromVector(x_initial)

simulator = Simulator(sim_diagram, sim_context)
# Use a smaller rollout step than the transcription internal step.
ResetIntegratorFromFlags(simulator, "runge_kutta2", min(h_sol / 20.0, 0.002))
simulator.set_target_realtime_rate(1.0)

meshcat.StartRecording()
simulator.Initialize()
simulator.AdvanceTo(rollout_times[-1])
meshcat.PublishRecording()

log = state_logger.FindLog(sim_context)
t_rollout = log.sample_times()
x_rollout = log.data()

x_final_rollout = x_rollout[:, -1]
print("rollout final state =", x_final_rollout)
print("rollout terminal error =", x_final_rollout - x_target)
print("rollout terminal error inf-norm =", np.linalg.norm(x_final_rollout - x_target, ord=np.inf))


In [ ]:
plt.figure()
plt.plot(tx, x_sol[0, :], "o-", markersize=2, label="NLP knot x")
plt.plot(t_rollout, x_rollout[0, :], label="rollout x")
plt.xlabel("time [s]")
plt.ylabel("cart position")
plt.grid(True)
plt.legend()

plt.figure()
plt.plot(tx, x_sol[1, :], "o-", markersize=2, label="NLP knot theta")
plt.plot(t_rollout, x_rollout[1, :], label="rollout theta")
plt.xlabel("time [s]")
plt.ylabel("pole angle")
plt.grid(True)
plt.legend()

plt.figure()
plt.plot(x_sol[0, :], x_sol[1, :], "o-", markersize=2, label="NLP knots")
plt.plot(x_rollout[0, :], x_rollout[1, :], label="rollout")
plt.xlabel("cart position")
plt.ylabel("pole angle")
plt.grid(True)
plt.legend()
plt.show()


## If the rollout is still off

Use these in order:

1. Increase `integration_substeps` from `6` to `10`.
2. Increase `N` from `81` to `121` or `161`.
3. Keep `h_max` smaller, e.g. `0.04`, so the zero-order-hold input is less coarse.
4. Warm-start the tighter problem from a looser terminal box.

The optimization terminal error and the Meshcat rollout error are not the same diagnostic. If `max abs defect` is tiny but the rollout drifts, the NLP is internally consistent; the open-loop input simply needs finer knot spacing or a rollout-stabilizing terminal controller.
